# Generación de descripciones comerciales de productos

Proyecto de destilación para descripciones en español. Se comparan soft prompt, LoRA y su combinación con dos conjuntos docentes, sobre las mismas 70 fichas exploratorias. El test permanece reservado.


## 1. Ejecución

El entrenamiento está desactivado por defecto. En Colab, selecciona una GPU T4, cambia `EJECUTAR` a `True` y carga el paquete autorizado. La ejecución genera archivos temporales y descarga un ZIP con resultados; la GPU debe desconectarse después de verificarlo.


In [ ]:
EJECUTAR = False

if EJECUTAR:
    import subprocess, sys, time
    INICIO_GLOBAL = time.monotonic()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.22.1', 'peft==0.21.2', 'accelerate==1.15.0', 'sacrebleu==2.6.0', 'pandas'], check=True)
    print('Dependencias instaladas')
else:
    print('Ejecución desactivada; no se instalaron paquetes ni se solicitó GPU o datos.')


## 2. Configuración

Se permite hasta 30 minutos por variante y dos horas en total. La parada temprana controla la pérdida docente, pero la validación contiene solo 34 ejemplos.


In [ ]:
if EJECUTAR:
    import gc, hashlib, json, math, random, shutil, time, zipfile
    from pathlib import Path
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from google.colab import files

    MODELO = 'Qwen/Qwen3-4B'
    REVISION = '1cfa9a7208912126459214e8b04321603b3df60c'
    SEMILLA = 20261031
    EPOCAS_MAX = 50
    PACIENCIA = 3
    MEJORA_MIN = 0.002
    MAX_SEGUNDOS_RAMA = 1800
    MAX_SEGUNDOS_TOTAL = 7200
    RESERVA_DESCARGA = 300
    LIMITE_TRABAJO = None if MAX_SEGUNDOS_TOTAL is None else MAX_SEGUNDOS_TOTAL - RESERVA_DESCARGA
    MAX_TOKENS_TOTAL = 1536
    TOKENS_SUAVES = 20
    PASOS_ACUMULACION = 4
    TASA_SOFT = 0.001
    TASA_LORA = 0.0001
    ESCALA_INICIAL = 128.0
    MAX_NUEVOS = 192
    EJEMPLOS_EVALUACION = 70
    ROOT = Path('/content/factorial_docente_rico_20261120')
    DATOS = ROOT / 'datos'
    RESULTADOS = ROOT / 'resultados'
    DATOS.mkdir(parents=True, exist_ok=True)
    RESULTADOS.mkdir(parents=True, exist_ok=True)
    assert torch.cuda.is_available(), 'Selecciona GPU T4'
    print('GPU:', torch.cuda.get_device_name(0), 'GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


## 3. Datos separados
Sube un ZIP con `train.jsonl`, `validacion_docente.jsonl`, `evaluacion.jsonl` y `manifest.json`. Cada fila requiere `product_locale`, `product_id`, `group_id`, `input` y `target`. La evaluación usa fichas separadas; su `target` puede ser una referencia original, no una verdad factual. El manifiesto debe incluir huellas, origen docente, decisión de aprobación y `test_included: false`.
El paquete utilizado se llama `paquete_lora_autorizado_20261120.zip`. Contiene 451 ejemplos de entrenamiento, 34 de validación docente y 70 fichas exploratorias. Los datos y pesos se distribuyen por separado; no vienen incluidos en el cuaderno.


In [ ]:
if EJECUTAR:
    carga = files.upload()
    assert len(carga) == 1 and next(iter(carga)).endswith('.zip')
    nombre = next(iter(carga))
    esperados = {'manifest.json', 'train.jsonl', 'validacion_docente.jsonl', 'evaluacion.jsonl'}
    with zipfile.ZipFile(nombre) as z:
        assert set(z.namelist()) == esperados
        manifest = json.loads(z.read('manifest.json'))
        assert manifest['test_included'] is False
        assert manifest['teacher_origin'] and manifest['approval'] == 'global_user_approval_filtered'
        for archivo in esperados - {'manifest.json'}:
            contenido = z.read(archivo)
            assert hashlib.sha256(contenido).hexdigest() == manifest['files'][archivo]['sha256']
            (DATOS / archivo).write_bytes(contenido)
    (DATOS / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))

    def cargar(nombre):
        marco = pd.read_json(DATOS / nombre, lines=True, dtype={'product_id': str, 'product_locale': str, 'group_id': str})
        columnas = ['product_locale', 'product_id', 'group_id', 'input', 'target']
        assert set(columnas).issubset(marco.columns) and len(marco) > 0
        assert marco[columnas].notna().all().all()
        assert all(marco[c].astype(str).str.strip().ne('').all() for c in columnas)
        assert not marco.duplicated(['product_locale', 'product_id']).any()
        return marco[columnas].copy()

    train = cargar('train.jsonl')
    val_docente = cargar('validacion_docente.jsonl')
    eval_todos = cargar('evaluacion.jsonl')
    eval_productos = eval_todos.head(EJEMPLOS_EVALUACION).copy()
    marcos = [train, val_docente, eval_todos]
    for i in range(len(marcos)):
        for j in range(i + 1, len(marcos)):
            assert not set(marcos[i].group_id) & set(marcos[j].group_id), 'Grupos cruzados entre particiones'
            assert not set(zip(marcos[i].product_locale, marcos[i].product_id)) & set(zip(marcos[j].product_locale, marcos[j].product_id)), 'Productos cruzados'
    print('Train:', len(train), 'validación docente:', len(val_docente), 'evaluación:', len(eval_productos))


## 4. Modelo e instrucción

Se utiliza Qwen3-4B con una revisión fijada y la misma instrucción para las tres variantes. Los vectores suaves preceden a la instrucción; LoRA ajusta adaptadores sobre los pesos originales.


In [ ]:
if EJECUTAR:
    from huggingface_hub import snapshot_download
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, TaskType, get_peft_model
    import transformers, peft

    INSTRUCCION = 'Redacta en español una descripción comercial clara y natural para el producto de la ficha. Identifica primero qué se vende y destaca las características útiles para decidir la compra: material, medidas con sus unidades, cantidad, compatibilidad, contenido y usos, cuando estén respaldados. Explica beneficios que se deduzcan directamente de esas características sin garantizar resultados. Conserva condiciones y restricciones importantes. Si la ficha contiene datos contradictorios, omite la afirmación dudosa o indica la discrepancia sin resolverla por tu cuenta. No inventes características, prestaciones, certificaciones ni accesorios. Mantén un tono vendedor sin exageraciones. Para fichas detalladas, desarrolla los datos más relevantes en varias frases; para fichas escasas, escribe solo lo que permitan los datos. No repitas datos para alcanzar una longitud ni añadas relleno. Entrega únicamente la descripción.'
    ruta_modelo = snapshot_download(MODELO, revision=REVISION)
    tok = AutoTokenizer.from_pretrained(ruta_modelo, local_files_only=True, trust_remote_code=False)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    def prefijo(texto):
        mensajes = [{'role': 'system', 'content': INSTRUCCION}, {'role': 'user', 'content': texto}]
        ids = tok.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True, enable_thinking=False)
        return ids['input_ids'] if hasattr(ids, 'keys') else ids

    def convertir(frame):
        ejemplos = []
        for fila in frame.itertuples(index=False):
            entrada = prefijo(fila.input)
            objetivo = tok.encode(fila.target, add_special_tokens=False) + [tok.eos_token_id]
            if len(entrada) + len(objetivo) + TOKENS_SUAVES > MAX_TOKENS_TOTAL:
                raise ValueError('Ejemplo excede el límite sin truncamiento: ' + fila.product_id)
            ejemplos.append({'product_id': fila.product_id, 'input_ids': entrada + objetivo, 'labels': [-100] * len(entrada) + objetivo})
        return ejemplos

    pares_train = convertir(train)
    pares_val = convertir(val_docente)
    for fila in eval_productos.itertuples(index=False):
        assert len(prefijo(fila.input)) + TOKENS_SUAVES + MAX_NUEVOS <= MAX_TOKENS_TOTAL, fila.product_id
    config = {'modelo': MODELO, 'revision': REVISION, 'semilla': SEMILLA, 'instruccion': INSTRUCCION, 'epocas_max': EPOCAS_MAX, 'paciencia': PACIENCIA, 'mejora_min': MEJORA_MIN, 'limite_segundos_por_rama': MAX_SEGUNDOS_RAMA, 'limite_segundos_total': MAX_SEGUNDOS_TOTAL, 'reserva_descarga_segundos': RESERVA_DESCARGA, 'tokens_suaves': TOKENS_SUAVES, 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'lora_targets': ['q_proj', 'v_proj'], 'tasa_soft': TASA_SOFT, 'tasa_lora': TASA_LORA, 'escala_inicial_fp16': ESCALA_INICIAL, 'acumulacion': PASOS_ACUMULACION, 'max_tokens_total': MAX_TOKENS_TOTAL, 'max_nuevos': MAX_NUEVOS, 'transformers': transformers.__version__, 'peft': peft.__version__, 'torch': torch.__version__}
    (RESULTADOS / 'configuracion.json').write_text(json.dumps(config, ensure_ascii=False, indent=2))
    print('Sin recortes:', len(pares_train), 'entrenamiento y', len(pares_val), 'validación docente')


## 5. Componentes compartidos
La pérdida se calcula solo sobre tokens docentes y se normaliza por el total del lote acumulado antes de escalarla en FP16. La escala inicial 128 reduce el riesgo de desbordamientos frente al valor predeterminado; no garantiza eliminarlos. Se registran los pasos saltados y los cambios de parámetros. Los checkpoints incluyen optimizador y aleatoriedad; no hay reanudación automática.
La detección de torchao se desactiva para este modelo FP16 sin cuantización por incompatibilidad del entorno de Colab.


In [ ]:
if EJECUTAR:
    import sys, peft.import_utils

    original_torchao = peft.import_utils.is_torchao_available
    for nombre, modulo in list(sys.modules.items()):
        if nombre.startswith('peft.') and modulo is not None and vars(modulo).get('is_torchao_available') is original_torchao:
            vars(modulo)['is_torchao_available'] = lambda: False

    def cargar_modelo(usar_lora):
        torch.manual_seed(SEMILLA)
        modelo = AutoModelForCausalLM.from_pretrained(ruta_modelo, dtype=torch.float16, device_map={'': 0}, local_files_only=True, trust_remote_code=False)
        modelo.config.use_cache = False
        for parametro in modelo.parameters():
            parametro.requires_grad_(False)
        if usar_lora:
            lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj', 'v_proj'])
            modelo = get_peft_model(modelo, lora)
        return modelo

    def crear_soft(modelo, activo):
        if not activo:
            return None
        origen = tok.encode(INSTRUCCION, add_special_tokens=False)
        ids = (origen * (TOKENS_SUAVES // len(origen) + 1))[:TOKENS_SUAVES]
        with torch.no_grad():
            inicial = modelo.get_input_embeddings()(torch.tensor(ids, device='cuda')).detach().float().clone()
        return torch.nn.Parameter(inicial)

    def embeddings(modelo, ids, soft):
        valores = modelo.get_input_embeddings()(ids)
        if soft is not None:
            valores = torch.cat((soft.to(valores.dtype).unsqueeze(0), valores), dim=1)
        return valores

    def perdida_suma(modelo, soft, ejemplo):
        ids = torch.tensor([ejemplo['input_ids']], device='cuda')
        labels = torch.tensor([ejemplo['labels']], device='cuda')
        if soft is not None:
            labels = torch.cat((torch.full((1, TOKENS_SUAVES), -100, device='cuda'), labels), dim=1)
        emb = embeddings(modelo, ids, soft)
        if modelo.training and not emb.requires_grad:
            emb.requires_grad_(True)
        salida = modelo(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device='cuda'), use_cache=False)
        logits = salida.logits[:, :-1, :].contiguous()
        objetivo = labels[:, 1:].contiguous()
        cantidad = int(objetivo.ne(-100).sum().item())
        assert cantidad > 0
        perdida = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), objetivo.view(-1), ignore_index=-100, reduction='sum')
        return perdida, cantidad

    def validar(modelo, soft):
        modelo.eval()
        suma, tokens = 0.0, 0
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            for ejemplo in pares_val:
                if time.monotonic() - INICIO_GLOBAL >= LIMITE_TRABAJO:
                    raise TimeoutError('Límite global durante la validación docente')
                valor, cantidad = perdida_suma(modelo, soft, ejemplo)
                suma += valor.item()
                tokens += cantidad
        return suma / tokens

    def guardar_estado(modelo, soft, opt, scaler, epoca, mejor, carpeta, rama):
        carpeta.mkdir(parents=True, exist_ok=True)
        if rama in {'lora', 'soft_lora'}:
            modelo.save_pretrained(carpeta / 'lora')
        if soft is not None:
            torch.save(soft.detach().float().cpu(), carpeta / 'soft.pt')
        torch.save({'optimizer': opt.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoca, 'best_val': mejor, 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all(), 'rng_python': random.getstate()}, carpeta / 'estado.pt')


## 6. Entrenamiento y parada temprana
Cada rama vuelve a cargar los pesos originales. El reloj es un límite adicional y puede interrumpir una época; se conserva el mejor checkpoint completado. Se registran pasos, tiempo, memoria y motivo de parada.

In [ ]:
if EJECUTAR:
    def entrenar(rama):
        usa_soft = rama in {'soft', 'soft_lora'}
        usa_lora = rama in {'lora', 'soft_lora'}
        random.seed(SEMILLA)
        torch.manual_seed(SEMILLA)
        torch.cuda.reset_peak_memory_stats()
        inicio = time.monotonic()
        modelo = cargar_modelo(usa_lora)
        soft = crear_soft(modelo, usa_soft)
        modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        grupos = []
        if usa_lora:
            grupos.append({'params': [p for p in modelo.parameters() if p.requires_grad], 'lr': TASA_LORA, 'name': 'lora'})
        if usa_soft:
            grupos.append({'params': [soft], 'lr': TASA_SOFT, 'name': 'soft'})
        assert grupos and all(g['params'] for g in grupos)
        iniciales = {g['name']: [p.detach().cpu().clone() for p in g['params']] for g in grupos}
        def grupos_modificados():
            return {g['name']: any(not torch.equal(p.detach().cpu(), antes) for p, antes in zip(g['params'], iniciales[g['name']])) for g in grupos}
        optimizador = torch.optim.AdamW(grupos, weight_decay=0.0)
        scaler = torch.amp.GradScaler('cuda', init_scale=ESCALA_INICIAL)
        mejor, sin_mejora, pasos, saltados, historial = float('inf'), 0, 0, 0, []
        salida = RESULTADOS / rama
        salida.mkdir(exist_ok=True)
        fin = 'max_epocas'
        for epoca in range(1, EPOCAS_MAX + 1):
            modelo.train()
            orden = list(range(len(pares_train)))
            random.shuffle(orden)
            optimizador.zero_grad(set_to_none=True)
            tokens_lote, suma_entrenamiento, total_tokens = 0, 0.0, 0
            for pos, indice in enumerate(orden, 1):
                if time.monotonic() - inicio >= MAX_SEGUNDOS_RAMA or time.monotonic() - INICIO_GLOBAL >= LIMITE_TRABAJO:
                    fin = 'limite_tiempo'
                    break
                if (pos - 1) % PASOS_ACUMULACION == 0:
                    lote = orden[pos - 1:pos - 1 + PASOS_ACUMULACION]
                    tokens_previstos = sum(sum(t != -100 for t in pares_train[k]['labels'][1:]) for k in lote)
                    assert tokens_previstos > 0
                with torch.autocast('cuda', dtype=torch.float16):
                    perdida, tokens = perdida_suma(modelo, soft, pares_train[indice])
                if not torch.isfinite(perdida):
                    raise RuntimeError('Pérdida no finita en ' + rama)
                scaler.scale(perdida / tokens_previstos).backward()
                tokens_lote += tokens
                suma_entrenamiento += perdida.item()
                total_tokens += tokens
                if pos % PASOS_ACUMULACION == 0 or pos == len(orden):
                    assert tokens_lote == tokens_previstos
                    scaler.unscale_(optimizador)
                    sin_gradiente = [g['name'] for g in optimizador.param_groups if not any(p.grad is not None for p in g['params'])]
                    if sin_gradiente:
                        raise RuntimeError('Grupo sin gradiente: ' + ', '.join(sin_gradiente))
                    entrenables = [p for grupo in optimizador.param_groups for p in grupo['params']]
                    finitos = all(torch.isfinite(p.grad).all().item() for p in entrenables if p.grad is not None)
                    if finitos:
                        torch.nn.utils.clip_grad_norm_(entrenables, 1.0)
                    escala_anterior = scaler.get_scale()
                    scaler.step(optimizador)
                    scaler.update()
                    if finitos and scaler.get_scale() >= escala_anterior:
                        pasos += 1
                        if pasos % 50 == 0:
                            print(rama, 'época', epoca, 'pasos', pasos, 'ejemplos', pos, '/', len(orden), flush=True)
                            guardar_estado(modelo, soft, optimizador, scaler, epoca, mejor, salida / 'intermedio', rama)
                    else:
                        saltados += 1
                        print(rama, 'paso omitido por gradientes no finitos', 'saltados:', saltados, 'escala:', scaler.get_scale(), flush=True)
                    optimizador.zero_grad(set_to_none=True)
                    tokens_lote = 0
            cambios = grupos_modificados()
            if fin == 'limite_tiempo':
                if pasos:
                    guardar_estado(modelo, soft, optimizador, scaler, epoca, mejor, salida / 'intermedio', rama)
                break
            if pasos == 0 or not all(cambios.values()):
                registro = {'epoca': epoca, 'pasos': pasos, 'pasos_saltados': saltados, 'loss_train': suma_entrenamiento / total_tokens, 'loss_val_docente': None, 'grupos_modificados': cambios, 'estado': 'sin_actualizaciones_validas', 'escala': scaler.get_scale(), 'segundos': time.monotonic() - inicio}
                historial.append(registro)
                (salida / 'historial.json').write_text(json.dumps(historial, ensure_ascii=False, indent=2))
                print(rama, registro, flush=True)
                continue
            guardar_estado(modelo, soft, optimizador, scaler, epoca, mejor, salida / 'intermedio', rama)
            val = validar(modelo, soft)
            if not math.isfinite(val):
                raise RuntimeError('Validación no finita en ' + rama)
            registro = {'epoca': epoca, 'pasos': pasos, 'pasos_saltados': saltados, 'loss_train': suma_entrenamiento / total_tokens, 'loss_val_docente': val, 'grupos_modificados': cambios, 'estado': 'validado', 'escala': scaler.get_scale(), 'segundos': time.monotonic() - inicio}
            historial.append(registro)
            (salida / 'historial.json').write_text(json.dumps(historial, ensure_ascii=False, indent=2))
            print(rama, registro, flush=True)
            if val < mejor - MEJORA_MIN:
                mejor, sin_mejora = val, 0
                guardar_estado(modelo, soft, optimizador, scaler, epoca, mejor, salida / 'mejor', rama)
            else:
                sin_mejora += 1
            guardar_estado(modelo, soft, optimizador, scaler, epoca, mejor, salida / 'ultimo', rama)
            if sin_mejora >= PACIENCIA:
                fin = 'parada_temprana'
                break
        cambios_finales = grupos_modificados()
        resumen = {'rama': rama, 'pasos': pasos, 'pasos_saltados': saltados, 'grupos_modificados': cambios_finales, 'escala_final': scaler.get_scale(), 'mejor_val_docente': mejor if math.isfinite(mejor) else None, 'motivo': fin, 'segundos': time.monotonic() - inicio, 'memoria_max_gib': torch.cuda.max_memory_allocated() / 2**30, 'epocas_completas': len(historial)}
        (salida / 'entrenamiento.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=2))
        if pasos == 0 or not all(cambios_finales.values()) or not math.isfinite(mejor):
            raise RuntimeError('No hay checkpoint validado con cambios reales en todos los grupos para ' + rama)
        del optimizador, scaler, modelo, soft
        gc.collect()
        torch.cuda.empty_cache()
        return resumen


## 7. Generación y métricas

Las tres variantes generan con configuración determinista. chrF++ mide similitud con la referencia; no mide veracidad ni calidad comercial. También se registran longitud y respuestas terminadas.


In [ ]:
if EJECUTAR:
    from peft import PeftModel
    from sacrebleu.metrics import CHRF
    import statistics

    def generar(modelo, soft, texto):
        ids = torch.tensor([prefijo(texto)], device='cuda')
        emb = embeddings(modelo, ids, soft)
        mascara = torch.ones(emb.shape[:2], dtype=torch.long, device='cuda')
        resultado = []
        modelo.eval()
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            salida = modelo(inputs_embeds=emb, attention_mask=mascara, use_cache=True)
            cache = salida.past_key_values
            siguiente = salida.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            for _ in range(MAX_NUEVOS):
                valor = int(siguiente.item())
                resultado.append(valor)
                if valor == tok.eos_token_id:
                    break
                mascara = torch.cat((mascara, torch.ones((1, 1), dtype=torch.long, device='cuda')), dim=1)
                salida = modelo(input_ids=siguiente, attention_mask=mascara, past_key_values=cache, use_cache=True)
                cache = salida.past_key_values
                siguiente = salida.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        return tok.decode(resultado, skip_special_tokens=True).strip(), bool(resultado and resultado[-1] == tok.eos_token_id)

    def evaluar(rama):
        torch.manual_seed(SEMILLA)
        modelo = cargar_modelo(False)
        mejor = RESULTADOS / rama / 'mejor'
        if rama in {'lora', 'soft_lora'}:
            modelo = PeftModel.from_pretrained(modelo, mejor / 'lora', is_trainable=False)
        soft = None
        if rama in {'soft', 'soft_lora'}:
            soft = torch.load(mejor / 'soft.pt', map_location='cuda', weights_only=True)
        filas = []
        for numero, fila in enumerate(eval_productos.itertuples(index=False), 1):
            if time.monotonic() - INICIO_GLOBAL >= LIMITE_TRABAJO:
                raise TimeoutError('Límite global; las predicciones parciales están guardadas')
            texto, eos = generar(modelo, soft, fila.input)
            filas.append({'product_locale': fila.product_locale, 'product_id': fila.product_id, 'group_id': fila.group_id, 'input': fila.input, 'reference': fila.target, 'prediction': texto, 'eos': eos})
            (RESULTADOS / (rama + '.json')).write_text(json.dumps(filas, ensure_ascii=False, indent=2))
            print(rama, numero, '/', len(eval_productos), flush=True)
        del modelo, soft
        gc.collect()
        torch.cuda.empty_cache()
        return filas

    def plantilla(texto):
        valores = []
        for linea in texto.splitlines():
            if ':' in linea:
                nombre, valor = linea.split(':', 1)
                if nombre.strip().lower() in {'titulo', 'marca', 'color', 'bullets'} and valor.strip():
                    valores.append(valor.strip())
        return '. '.join(valores)

    def metricas(filas):
        referencias = [x['reference'] for x in filas]
        return {'n': len(filas), 'eos': sum(x['eos'] for x in filas), 'palabras_media': statistics.mean(len(x['prediction'].split()) for x in filas), 'palabras_mediana': statistics.median(len(x['prediction'].split()) for x in filas), 'chrf_plus_plus': CHRF(word_order=2).corpus_score([x['prediction'] for x in filas], [referencias]).score, 'plantilla_chrf_plus_plus': CHRF(word_order=2).corpus_score([plantilla(x['input']) for x in filas], [referencias]).score}


## 8. Entrenamiento y evaluación

Las variantes se entrenan y evalúan secuencialmente. Se conservan el mejor punto validado y los resultados disponibles; los límites pueden dejar ramas incompletas.


In [ ]:
if EJECUTAR:
    todas = {}
    fallo = None
    try:
        if not isinstance(MAX_SEGUNDOS_RAMA, (int, float)) or not isinstance(MAX_SEGUNDOS_TOTAL, (int, float)) or MAX_SEGUNDOS_RAMA <= 0 or MAX_SEGUNDOS_TOTAL <= RESERVA_DESCARGA:
            raise ValueError('Configura límites de GPU expresamente autorizados')
        for rama in ['soft', 'lora', 'soft_lora']:
            if time.monotonic() - INICIO_GLOBAL >= LIMITE_TRABAJO:
                raise TimeoutError('Reserva de descarga alcanzada')
            if rama != 'raw':
                entrenar(rama)
            salidas = evaluar(rama)
            todas[rama] = metricas(salidas)
            (RESULTADOS / 'metricas.json').write_text(json.dumps(todas, ensure_ascii=False, indent=2))
            print('Resultado:', rama, todas[rama], flush=True)
    except Exception as error:
        fallo = {'tipo': type(error).__name__, 'mensaje': str(error)}
        print('Ejecución incompleta:', fallo, flush=True)
    finally:
        estado = {'estado': 'completa' if fallo is None else 'incompleta', 'error': fallo, 'ramas_evaluadas': list(todas), 'segundos_totales_antes_de_empaquetar': time.monotonic() - INICIO_GLOBAL}
        (RESULTADOS / 'estado.json').write_text(json.dumps(estado, ensure_ascii=False, indent=2))
        destino = ROOT / 'factorial_docente_rico_resultados_20261120.zip'
        with zipfile.ZipFile(destino, 'w', compression=zipfile.ZIP_DEFLATED) as z:
            for archivo in RESULTADOS.rglob('*'):
                if archivo.is_file():
                    z.write(archivo, archivo.relative_to(ROOT))
        print('SHA-256:', hashlib.sha256(destino.read_bytes()).hexdigest(), flush=True)
        files.download(str(destino))


## 9. Resultados y límites

El ZIP descargado contiene las salidas y el estado de la ejecución. La evaluación humana y el análisis de respaldo factual se realizan por separado.


In [ ]:
if EJECUTAR:
    print('Consulta estado.json y conserva el ZIP descargado antes de cerrar la sesión')


## 10. Comparación exploratoria: maestro anterior y nuevo

Los resultados anteriores proceden de la ejecución del 18/11/2026 con 2.268 ejemplos docentes de entrenamiento y 188 de validación. La ejecución nueva utiliza 451 y 34, respectivamente. Cambian el docente, el tamaño de los datos y el presupuesto de entrenamiento, por lo que esta comparación no aísla el efecto del docente. Las 70 fichas exploratorias son las mismas; no son test final.

Fuente histórica: `artifacts/docente_comercial_20261117/resultados_colab_20261118/factorial_docente_escala_resultados.zip`, SHA-256 `78b6e14baccd5d1f763955c472c796de20c76106ca63ce823804319f88f1bbcd`. El ZIP no se presupone incluido en GitHub.


In [ ]:
import json
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
from IPython.display import display

historico = {'soft': {'n': 70, 'eos': 70, 'palabras_media': 33.371428571428574, 'palabras_mediana': 34.0, 'chrf_plus_plus': 11.025291833690947, 'plantilla_chrf_plus_plus': 36.913927288329994}, 'lora': {'n': 70, 'eos': 70, 'palabras_media': 25.428571428571427, 'palabras_mediana': 25.0, 'chrf_plus_plus': 8.432358337297773, 'plantilla_chrf_plus_plus': 36.913927288329994}, 'soft_lora': {'n': 70, 'eos': 70, 'palabras_media': 24.12857142857143, 'palabras_mediana': 23.5, 'chrf_plus_plus': 7.983136404014777, 'plantilla_chrf_plus_plus': 36.913927288329994}}
nombres = {'soft': 'Soft prompt', 'lora': 'LoRA', 'soft_lora': 'Soft prompt + LoRA'}
filas = [{'variante': nombres[k], 'docente': 'Anterior', 'n': v['n'], 'chrF++': v['chrf_plus_plus'], 'palabras promedio': v['palabras_media'], 'terminadas': v['eos']} for k, v in historico.items()]
resultados_nuevos = ''
if resultados_nuevos:
    with ZipFile(Path(resultados_nuevos)) as archivo:
        nuevas = json.loads(archivo.read('resultados/metricas.json'))
        estado = json.loads(archivo.read('resultados/estado.json'))
    for k in nombres:
        if k in nuevas:
            v = nuevas[k]
            filas.append({'variante': nombres[k], 'docente': 'Nuevo', 'n': v['n'], 'chrF++': v['chrf_plus_plus'], 'palabras promedio': v['palabras_media'], 'terminadas': v['eos']})
    print('Estado de la ejecución nueva:', estado)
else:
    print('Resultados del docente nuevo pendientes de incorporar; indica la ruta de su ZIP en resultados_nuevos.')
display(pd.DataFrame(filas).round(2))


## 11. Evaluación humana

Diez fichas se presentarán a personas ajenas al proyecto con seis descripciones anónimas: las tres variantes con cada docente. Cada persona ordenará las respuestas y señalará afirmaciones sin respaldo. La muestra será exploratoria.


## 12. Reproducibilidad y alcance

Los datos docentes se aprobaron globalmente y se filtraron con revisión asistida por IA; no se atribuye revisión humana individual. Las particiones y el test permanecen separados. Las referencias pueden contener información ausente de la ficha, por lo que ni chrF++ ni la pérdida docente sustituyen una revisión factual. Los resultados y adaptadores se obtienen por separado; no están incluidos en el cuaderno.
